In [1]:
from pyspark.sql.functions import col, min, max, count, sum , when, round

StatementMeta(, 40ad881f-407f-428d-9b06-559aad4e0566, 3, Finished, Available, Finished, False)

In [2]:
## Read silver files 
df_cust = spark.read.table("lh_insurance_silver.dbo.dim_customers")
df_pol = spark.read.table("lh_insurance_silver.dbo.dim_policies")
df_claims = spark.read.table("lh_insurance_silver.dbo.fact_claims")


StatementMeta(, 40ad881f-407f-428d-9b06-559aad4e0566, 4, Finished, Available, Finished, False)

In [4]:
## Enrich Fact Table with Foreign Keys & Dimensions

df_fact_claims = df_claims \
        .join(df_pol.select("policy_id", "cust_id", "policy_type", "coverage_amount"), on="policy_id", how="left")\
        .select(
            col("claim_id"),
            col("policy_id"),
            col("cust_id"),
            col("claim_date"),
            col("claim_amount"),
            col("claim_status"),
            col("policy_type"),
            col("coverage_amount"),
            when(col("claim_status") == "APPROVED", 1).otherwise(0).alias("is_approved"),
            when(col("claim_status") == "REJECTED", 1).otherwise(0).alias("is_rejected")
        )


StatementMeta(, 2616c4df-092f-4115-8981-94572cf4cb0b, 6, Finished, Available, Finished, False)

In [5]:
gold_table_path = "abfss://insurance@onelake.dfs.fabric.microsoft.com/lh_insurance_gold.Lakehouse/Tables/dbo/gold_fct_claims"

df_fact_claims.write\
.format("delta")\
.mode("overwrite")\
.save(gold_table_path)

StatementMeta(, 2616c4df-092f-4115-8981-94572cf4cb0b, 7, Finished, Available, Finished, False)

In [9]:
## Customer Summary Aggregation (Top Claimants, Tenure, Risk Ratio)

df_cust_agg = df_fact_claims.groupBy("cust_id").agg(
    count("claim_id").alias("total_claims"),
    sum("claim_amount").alias("total_claim_amount"),
    min("claim_date").alias("first_claim_date"),
    max("claim_date").alias("last_claim_date")
)


StatementMeta(, 2616c4df-092f-4115-8981-94572cf4cb0b, 11, Finished, Available, Finished, False)

In [12]:
df_cust_kpi = df_cust.join(df_cust_agg, on = "cust_id", how = "left")\
                .join(df_pol.groupBy("cust_id").agg(sum("coverage_amount").alias("total_coverage_amount")),
                on = "cust_id", 
                how = 'left')\
                .withColumn(
                    "claim_to_coverage_ratio", 
        round(col("total_claim_amount") / col("total_coverage_amount"), 4)
        
        )



StatementMeta(, 2616c4df-092f-4115-8981-94572cf4cb0b, 16, Finished, Available, Finished, False)

In [13]:
## Save Dimensions

gold_table_path = "abfss://insurance@onelake.dfs.fabric.microsoft.com/lh_insurance_gold.Lakehouse/Tables/dbo/gold_dim_customer"


df_cust.write\
.format("delta")\
.mode("overwrite")\
.save(gold_table_path)


StatementMeta(, 2616c4df-092f-4115-8981-94572cf4cb0b, 17, Finished, Available, Finished, False)

In [15]:
gold_table_path = "abfss://insurance@onelake.dfs.fabric.microsoft.com/lh_insurance_gold.Lakehouse/Tables/dbo/gold_dim_policy"


df_pol.write\
.format("delta")\
.mode("overwrite")\
.save(gold_table_path)

StatementMeta(, 2616c4df-092f-4115-8981-94572cf4cb0b, 19, Finished, Available, Finished, False)

In [2]:
## date table creation

from pyspark.sql.functions import col, explode, to_date, year, quarter, month, date_format, dayofmonth, concat, lit

df_dates = spark.sql("SELECT sequence(to_date('2015-01-01'), to_date('2030-12-31'), interval 1 day) as date_seq") \
    .withColumn("date", explode(col("date_seq"))) \
    .select(
        col("date"),
        year("date").alias("year"),
        quarter("date").alias("quarter_number"),
        concat(lit("Q"), quarter("date")).alias("quarter"),
        month("date").alias("month_number"),
        date_format("date", "MMMM").alias("month_name"),
        date_format("date", "MMM").alias("month_short"),
        dayofmonth("date").alias("day_of_month"),
        date_format("date", "yyyy-MM").alias("year_month"),
        date_format("date", "EEEE").alias("day_name")
    )

StatementMeta(, 9cbaade4-b45a-4be2-8c0e-f23e097bc1a4, 4, Finished, Available, Finished, False)

In [3]:
date_table_path = "abfss://insurance@onelake.dfs.fabric.microsoft.com/lh_insurance_gold.Lakehouse/Tables/dbo/gold_dim_date"


df_dates.write \
    .format("delta") \
    .mode("overwrite") \
    .save(date_table_path)

StatementMeta(, 9cbaade4-b45a-4be2-8c0e-f23e097bc1a4, 5, Finished, Available, Finished, False)